# ICE Outcomes: Data Cleaning, Feature Engineering, and EDA

In [ ]:
# imports
import pandas as pd
import numpy as np
import re

In [ ]:
# Load the CSV from your local path
df = pd.read_csv("../data/ICE_Master_Clean.csv")

# Print all column names
print("Column Names:")
print(df.columns.tolist())

# Total column count
print(f"\nTotal columns: {len(df.columns)}")

# Column names with data types
print("\nColumn names and data types:")
print(df.dtypes)

# Check for duplicate column names
dupes = df.columns[df.columns.duplicated()].tolist()
print(f"\nDuplicate columns: {dupes if dupes else 'None'}")

# Check for unnamed/blank columns
unnamed = [col for col in df.columns if 'Unnamed' in str(col) or str(col).strip() == '']
print(f"Unnamed/blank columns: {unnamed if unnamed else 'None'}")

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_3296/3372129018.py:4: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,21,22,23,29,30,31,32,33,34,35,36,37,38,39,40,42,45,46,47,48,49,50,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,69,70,71,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,90,91,92,93,94,96,97,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/ICE_Master_Clean.csv")


Column Names:
['Anonymized Identifier', 'Apprehension Date', 'Apprehension Method', 'Arrest Created By', 'Case ID (arrests)', 'Subject ID (arrests)', 'Alien File Number (arrests)', 'RCA_AOR', 'RCA_DCO', 'A_NUMBER', 'SUBJ_ID', 'LAST_NAME', 'FIRST_NAME', 'ALERT_CODE', 'RISK_TO_PUBLIC_SAFETY', 'RISK_OF_FLIGHT', 'SPECIAL_VULNERABILITY', 'CASE_CAT_AT_RCA_DECISION', 'REMOVAL_LIKELY_AT_RCA_DECISION', 'RCA_RECOMMENDATION', 'RCA_BOND_RECOMMENDATION', 'OFFICER_ID', 'SUPERVISOR_ID', 'RCA_FINAL_DECISION', 'FINAL_BOND_AMOUNT', 'SPEC_VULN_VER', 'MAN_DET_VER', 'DISC_INFR_VER', 'fiscal_year_code', 'fiscal_quarter_name', 'Detention ID', 'Case ID (detentions)', 'Subject ID (detentions)', 'Detention Book In Date', 'Detention Facility', 'Religion', 'Marital', 'Gender', 'Birth Date (detentions)', 'Ethnicity', 'Alien File Number (detentions)', 'Birth Year', 'Entry Status', 'Bond Posted Amount', 'Initial Bond Set Amount', 'Case Status', 'Case Category', 'Final Order Yes No (TARGET)', 'Final Order Date (deten

In [3]:
df.head()

,Anonymized Identifier,Apprehension Date,Apprehension Method,Arrest Created By,Case ID (arrests),Subject ID (arrests),Alien File Number (arrests),RCA_AOR,RCA_DCO,A_NUMBER,...,Felon,Book In Criminality,Final Charge,Citizenship Country (detentions),Final Program (detentions),MSC Charge Code (detentions),MSC Charge (detentions),administration,final_order_binary,age_at_arrest
0,d0451cac01101d02e3b5c236e6b7432e5eff2148,2012-02-29,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Obama,0,43.0
1,722038e8783f0111938ba418485b72719130bb42,2012-06-12,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Obama,0,33.0
2,f196886fb97872592bce7642b1c1ea79722672b5,2012-08-23,CAP Local Incarceration,(b)(6)(b)(7)(c),NaN,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Obama,0,38.0
3,764bb6cc37b909a941aeebe9ae5f813bfe450c87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Obama,0,38.0
4,c269a353ec849af9c41c95409e30fe9b960889c7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Obama,0,39.0


In [4]:
print(df['administration'].value_counts())

administration
Trump    1024770
Obama     935939
Biden     549014
Name: count, dtype: int64


### Step 1
In our case, we want to predict whether someone received a final deportation order. The column final_order_binary already encodes this as 1 (yes, deported) or 0 (no). We also drop any rows where this value is missing, because a row with no known outcome can't be used to train or evaluate a model.

In [ ]:
# STEP 1: TARGET VARIABLE

# --- Fix the target variable ---
# final_order_binary is broken — it contains only 0s and cannot be used.
# The correct target is 'Final Order Yes No (TARGET)' which contains
# True/False values representing whether someone received a final
# deportation order. We convert True → 1 and False → 0 ourselves.

print("Raw target column:")
print(df['Final Order Yes No (TARGET)'].value_counts(dropna=False))

# Convert True/False to 1/0
df['target'] = df['Final Order Yes No (TARGET)'].map({True: 1, False: 0})

print(f"\nConverted target distribution:")
print(df['target'].value_counts())
print(f"Missing: {df['target'].isna().sum():,}")
print(f"\nDeportation rate: {df['target'].mean():.1%}")

# Drop rows where target is missing — we can't learn from those
df = df[df['target'].notna()].copy()
y = df['target'].astype(int)

print(f"\nRows remaining after dropping missing targets: {len(df):,}")
print(f"Final deportation rate: {y.mean():.1%}")

Raw target column:
Final Order Yes No (TARGET)
True     1681397
False     618100
NaN       210226
Name: count, dtype: int64

Converted target distribution:
target
1.0    1681397
0.0     618100
Name: count, dtype: int64
Missing: 210,226

Deportation rate: 73.1%

Rows remaining after dropping missing targets: 2,299,497
Final deportation rate: 73.1%


### Step 2
This dataset was obtained through a Freedom of Information Act (FOIA) request, which means the government released it to the public but was legally required to black out certain personal identifying information. Instead of actually blacking out the text, the dataset uses the code (b)(6)(b)(7)(c) wherever information was withheld. This shows up all over columns like names, case IDs, and alien file numbers. These values look like real data but are actually just a placeholder meaning "redacted." If we left them in, the model might treat them as a meaningful category, which would be misleading. So we replace every instance of this redaction code with a true missing value (NaN) so Python knows the data isn't there.


In [8]:
# STEP 2: HANDLE FOIA REDACTIONS
foia_pattern = r'\(b\)\(\d+\)'

# Only look at object columns, but explicitly cast to string first
# because some "object" columns contain mixed types (numbers + text)
# which causes .str to fail
obj_cols = df.select_dtypes(include='object').columns

for col in obj_cols:
    # Convert to string temporarily just for the check
    # na=False means NaN values are ignored and treated as non-matches
    col_as_str = df[col].astype(str)
    mask = col_as_str.str.contains(foia_pattern, regex=True, na=False)
    if mask.sum() > 0:
        df.loc[mask, col] = np.nan
        print(f"  Replaced {mask.sum()} redacted values in: {col}")

print(f"\nDataset shape after FOIA cleaning: {df.shape}")


  Replaced 960809 redacted values in: Arrest Created By
  Replaced 1070148 redacted values in: Case ID (arrests)
  Replaced 1077326 redacted values in: Subject ID (arrests)
  Replaced 1077326 redacted values in: Alien File Number (arrests)
  Replaced 838321 redacted values in: A_NUMBER
  Replaced 838321 redacted values in: SUBJ_ID
  Replaced 838321 redacted values in: LAST_NAME
  Replaced 838317 redacted values in: FIRST_NAME
  Replaced 722871 redacted values in: OFFICER_ID
  Replaced 819049 redacted values in: SUPERVISOR_ID
  Replaced 1903919 redacted values in: Detention ID
  Replaced 2183933 redacted values in: Case ID (detentions)
  Replaced 2183933 redacted values in: Subject ID (detentions)
  Replaced 2299497 redacted values in: Birth Date (detentions)
  Replaced 2183933 redacted values in: Alien File Number (detentions)
  Replaced 1084734 redacted values in: Case ID (removals)
  Replaced 1084734 redacted values in: Birth Date (removals)
  Replaced 1084734 redacted values in: Ali

### Step 3
Data leakage is one of the most important concepts in machine learning, and it's easy to accidentally get wrong. Leakage happens when your model gets access to information that it wouldn't actually have at the time it needs to make a prediction. In our case, we're trying to predict whether someone will receive a deportation order. But several columns in this dataset — like departure dates, final order dates, and removal records — describe what happened after that order was already issued. If we let the model see those, it's essentially cheating: it's learning from the answer, not from the inputs. We also remove final_order_binary from the feature set here since it is the answer — we already saved it separately as y.

In [ ]:
# STEP 3: DROP LEAKAGE COLUMNS
# Anything that occurs AFTER the deportation order
leakage_cols = [
    # All removals columns — these record what happened after the order
    'Final Order Yes No (removals)', 'Final Order Date (removals)',
    'Departure Date (removals)', 'Departure Country (removals)',
    'Port of Departure', 'Entry Status (removals)',
    'Birth Date (removals)', 'Alien File Number (removals)',
    'Case ID (removals)', 'Processing Disposition',
    'Processing Disposition Code', 'Current Program',

    # Detention outcome columns — also post-order
    'Final Order Yes No (TARGET)',
    'Final Order Date (detentions)',
    'Departed Date (detentions)',
    'Departure Country (detentions)',

    # Arrest outcome columns — post-order
    'Final Order Yes No (arrests)',
    'Final Order Date (arrests)',
    'Departed Date (arrests)',
    'Departure Country (arrests)',

    # Other post-order outcome fields
    'Final Order Yes No (detainers)',
    'Deportation Ordered Yes No',

    # This is our target variable — belongs in y, not X
    'final_order_binary',
]

# Only drop columns that actually exist in the dataframe
leakage_cols = [c for c in leakage_cols if c in df.columns]
df = df.drop(columns=leakage_cols)

print(f"Columns remaining after removing leakage: {df.shape[1]}")

Columns remaining after removing leakage: 100


### Step 4
These columns either contain personally identifiable information (PII) like names, or are identifier columns like case IDs and file numbers. Neither type is useful for prediction. Names and IDs are unique to each individual — a model can't learn anything generalizable from them, and they'd also raise serious ethical concerns if included. Officer and supervisor IDs fall into the same category: they're specific to individual employees and wouldn't generalize to new data. Most of these columns were also heavily redacted in Step 2, so they're largely empty at this point anyway.


In [ ]:
# STEP 4: DROP ID / PII / REDACTED COLUMNS
# No predictive value; largely redacted anyway
id_cols = [
    'Anonymized Identifier',
    'Case ID (arrests)', 'Case ID (detentions)',
    'Subject ID (arrests)', 'Subject ID (detentions)',
    'Alien File Number (arrests)', 'Alien File Number (detentions)',
    'Alien File Number (removals)',
    'A_NUMBER', 'SUBJ_ID',
    'FIRST_NAME', 'LAST_NAME',
    'OFFICER_ID', 'SUPERVISOR_ID',
    'Arrest Created By',
    'Detention ID',
]

id_cols = [c for c in id_cols if c in df.columns]
df = df.drop(columns=id_cols)

print(f"Columns remaining after removing IDs and PII: {df.shape[1]}")

Columns remaining after removing IDs and PII: 85


### Step 5
Several columns in this dataset contain open-ended written comments from officers and supervisors — things like notes on special vulnerabilities or reasons a removal was considered unlikely. While these might sound informative, they're extremely difficult to use in a standard machine learning model because they're unstructured text. They're also largely empty (most rows have no comment at all), and they may reflect individual officer bias in ways that are hard to account for. Analyzing free text would require a separate natural language processing (NLP) pipeline, which is beyond the scope of this study.

In [ ]:
# STEP 5: DROP FREE-TEXT COMMENT COLUMNS
comment_cols = [
    'SPECIAL_VULNERABILITY_COMMENTS',
    'REASON_REMOVAL_UNLIKELY_AT_RCA_DECISION',
    'OFFICER_COMMENTS',
    'SUPERVISOR_COMMENTS',
]

comment_cols = [c for c in comment_cols if c in df.columns]
df = df.drop(columns=comment_cols)

print(f"Columns remaining after removing comment columns: {df.shape[1]}")

Columns remaining after removing comment columns: 81


### Step 6
Because this dataset merges records from three different ICE systems — arrests, detentions, and removals — many pieces of information appear more than once under slightly different column names. For example, birth date appears as Birth Date (arrests), Birth Date (detentions), and Birth Date (removals). Keeping all three would confuse the model and inflate the importance of that variable. We keep the detention version of shared fields because the detention record is the most complete and central stage of the enforcement process for our research question.

In [ ]:
# STEP 6: DROP REDUNDANT DUPLICATE COLUMNS
# Where same info appears across arrests/detentions/removals,
# keep the detention version as it's most complete
redundant_cols = [
    # Birth date — keeping detentions version
    'Birth Date (arrests)', 'Birth Date (removals)',
    'Birth Year (arrests)',

    # Citizenship and gender — keeping main columns
    'Citizenship Country (arrests)',
    'Gender (arrests)',

    # Case info — keeping detentions version
    'Case Status (arrests)', 'Case Category (arrests)',
    'Final Program (detentions)',

    # Criminal charge info — keeping main versions
    'MSC Charge (detainers)',
    'MSC Charge Code (detentions)', 'MSC Charge (detentions)',

    # Case threat — keeping main version
    'Case Threat Level (removals)',

    # Apprehension criminality — keeping Book In Criminality
    # because it reflects criminality at the point of detention,
    # which is more directly relevant to deportation decisions
    'Apprehension Criminality',
]

redundant_cols = [c for c in redundant_cols if c in df.columns]
df = df.drop(columns=redundant_cols)

print(f"Columns remaining after removing redundant duplicates: {df.shape[1]}")

Columns remaining after removing redundant duplicates: 69


### Step 7
Even after all the drops above, some columns may still be mostly empty. This can happen because not every individual went through every stage of the ICE process — for example, someone who was arrested but never formally detained won't have detention-specific fields filled in. A column that is more than 60% empty is generally not reliable enough to train a model on, so we remove those here. The 60% threshold is a common rule of thumb — you can raise or lower it, but going much higher risks feeding the model columns that are almost entirely guesswork.

In [ ]:
# STEP 7: DROP HIGH-MISSING COLUMNS (>60%)
# After FOIA cleaning, reassess true missingness
missing_rate = df.isnull().mean()
high_missing = missing_rate[missing_rate > 0.60].index.tolist()

print(f"Dropping {len(high_missing)} columns with more than 60% missing values:")
print(high_missing)

df = df.drop(columns=high_missing)
print(f"\nColumns remaining after missingness drop: {df.shape[1]}")

Dropping 40 columns with more than 60% missing values:
['ALERT_CODE', 'CASE_CAT_AT_RCA_DECISION', 'REMOVAL_LIKELY_AT_RCA_DECISION', 'RCA_BOND_RECOMMENDATION', 'FINAL_BOND_AMOUNT', 'MAN_DET_VER', 'DISC_INFR_VER', 'Religion', 'Birth Date (detentions)', 'Bond Posted Amount', 'Initial Bond Set Amount', 'MSC Charge', 'MSC Charge Code', 'MSC Conviction Date', 'MSC Criminal Charge Status', 'Apprehension State', 'Apprehension AOR', 'Final Program', 'Final Program Group', 'Apprehension Site Landmark', 'Facility State', 'Detainer Prepared Criminality', 'Detainer Prep Threat Level', 'Birth Country (detainers)', 'MSC Sentence Days', 'MSC Sentence Months', 'MSC Sentence Years', 'Felon (detainers)', 'Prior Felony Yes No', 'Violent Misdemeanor Yes No', 'Illegal Entry Yes No', 'Illegal Reentry Yes No', 'Immigration Fraud Yes No', 'Significant Risk Yes No', 'Criminal Street Gang Yes No', 'Aggravated Felony Yes No', 'Felon', 'Book In Criminality', 'Final Charge', 'Citizenship Country (detentions)']

Col

In [ ]:
# Let's see exactly what 29 columns we have left, their data types,
# and how much missing data remains — this is our actual working dataset

print(f"Dataset shape: {df.shape}")
print(f"\nAll remaining columns and their dtypes:")
print(df.dtypes)

print(f"\nMissing values per column:")
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False))

print(f"\nTarget distribution:")
print(y.value_counts())
print(f"Deportation rate: {y.mean():.1%}")

Dataset shape: (2299497, 29)

All remaining columns and their dtypes:
Apprehension Date          object
Apprehension Method        object
RCA_AOR                    object
RCA_DCO                    object
RISK_TO_PUBLIC_SAFETY      object
RISK_OF_FLIGHT             object
SPECIAL_VULNERABILITY      object
RCA_RECOMMENDATION         object
RCA_FINAL_DECISION         object
SPEC_VULN_VER             float64
fiscal_year_code          float64
fiscal_quarter_name        object
Detention Book In Date     object
Detention Facility         object
Marital                    object
Gender                     object
Ethnicity                  object
Birth Year                float64
Entry Status               object
Case Status                object
Case Category              object
Case Threat Level         float64
Charge                     object
Birth Country              object
Citizenship Country        object
year                        int64
administration             object
age_at_arres

In [ ]:
# This tells us whether the missing demographic data is random
# or systematic — if certain administrations or outcomes have
# more missing ethnicity/citizenship data, that's meaningful
# for your paper's argument about data transparency

print("=== MISSINGNESS PATTERNS ON KEY VARIABLES ===\n")

key_vars = ['Ethnicity', 'Citizenship Country', 'Birth Country',
            'Apprehension Method', 'Case Threat Level']

for var in key_vars:
    print(f"\n--- {var} ---")
    # Missing rate by administration
    miss_by_admin = df.groupby('administration')[var].apply(
        lambda x: x.isna().mean()
    ).round(3)
    print(f"Missing rate by administration:")
    print(miss_by_admin)

print("\n=== VALUE COUNTS ON KEY DEMOGRAPHIC VARIABLES ===\n")

# What values do we actually have for ethnicity and citizenship?
for var in ['Ethnicity', 'Gender', 'Entry Status', 'Apprehension Method']:
    if var in df.columns:
        print(f"\n{var}:")
        print(df[var].value_counts(dropna=False).head(10))

=== MISSINGNESS PATTERNS ON KEY VARIABLES ===


--- Ethnicity ---
Missing rate by administration:
administration
Biden    0.684
Obama    0.535
Trump    0.461
Name: Ethnicity, dtype: float64

--- Citizenship Country ---
Missing rate by administration:
administration
Biden    0.684
Obama    0.430
Trump    0.564
Name: Citizenship Country, dtype: float64

--- Birth Country ---
Missing rate by administration:
administration
Biden    0.684
Obama    0.430
Trump    0.564
Name: Birth Country, dtype: float64

--- Apprehension Method ---
Missing rate by administration:
administration
Biden    0.812
Obama    0.539
Trump    0.428
Name: Apprehension Method, dtype: float64

--- Case Threat Level ---
Missing rate by administration:
administration
Biden    0.697
Obama    0.411
Trump    0.461
Name: Case Threat Level, dtype: float64

=== VALUE COUNTS ON KEY DEMOGRAPHIC VARIABLES ===


Ethnicity:
Ethnicity
NaN                       1207042
Hispanic Origin            991521
Not of Hispanic Origin      8707

### Step 8 — Feature Engineering
This step transforms raw columns into variables that are actually useful for a model to learn from. We do four things.

Dates. Instead of feeding raw date strings into the model, we extract the number of days between apprehension and detention booking, the month of apprehension, and the year. These capture processing speed, seasonal patterns, and policy era.

Criminal history. The detailed criminal history columns (prior felony, aggravated felony, etc.) were 88–100% missing. Rather than dropping this information entirely or imputing mostly made-up values, we go back to the raw data and ask two simpler questions: did ICE record any criminality for this person at any stage (any_criminality_recorded: 0 or 1), and if so, what was the most serious level (max_criminality_severity: 0–3 scale from nothing recorded up to convicted criminal). This is our primary way of testing Hypothesis 1 given the data limitations, and we note it as such in the methods section.

Apprehension method. We create two binary flags from the raw text column: whether the arrest was a CAP arrest (person was already in criminal custody when ICE picked them up) and whether it involved a 287(g) partnership with local law enforcement. Both directly measure the criminal justice and immigration entanglement your paper argues about.

Demographics. We simplify Ethnicity, Gender, and Entry Status into three binary flags — Hispanic origin, male, and Present Without Authorization — rather than one-hot encoding them into many columns.

In [ ]:
# STEP 8: FEATURE ENGINEERING

# --- Parse date columns (only if they still exist) ---
# We use 'if in df.columns' checks throughout because if you
# re-run this cell, some columns may already be dropped from
# a previous partial run

if 'Apprehension Date' in df.columns and 'Detention Book In Date' in df.columns:
    df['Apprehension Date'] = pd.to_datetime(df['Apprehension Date'], errors='coerce')
    df['Detention Book In Date'] = pd.to_datetime(df['Detention Book In Date'], errors='coerce')
    df['days_to_book_in'] = (
        df['Detention Book In Date'] - df['Apprehension Date']
    ).dt.days
    df['apprehension_month'] = df['Apprehension Date'].dt.month
    df['apprehension_year']  = df['Apprehension Date'].dt.year
    df = df.drop(columns=['Apprehension Date', 'Detention Book In Date'])
    print("✅ Date features created")
else:
    print("⚠️  Date columns already processed or not present — skipping")

# ---------------------------------------------------------------
# CRIMINAL HISTORY FEATURES
# ---------------------------------------------------------------
df_raw = pd.read_csv("/Users/sabrinkulatein/Downloads/ICE_Master_Clean.csv")
df_raw_aligned = df_raw.loc[df.index]

print(f"df rows: {len(df):,} | df_raw_aligned rows: {len(df_raw_aligned):,}")

if 'any_criminality_recorded' not in df.columns:
    criminal_cols_raw = [
        'Apprehension Criminality',
        'Book In Criminality',
        'Detainer Prepared Criminality',
    ]
    df['any_criminality_recorded'] = (
        df_raw_aligned[criminal_cols_raw]
        .notna()
        .any(axis=1)
        .astype(int)
    )

    severity_map = {
        '1 Convicted Criminal': 3,
        '2 Pending Criminal Charges': 2,
        '3 Other Immigration Violator': 1,
    }
    severity_scores = []
    for col in criminal_cols_raw:
        if col in df_raw_aligned.columns:
            severity_scores.append(df_raw_aligned[col].map(severity_map))

    df['max_criminality_severity'] = (
        pd.concat(severity_scores, axis=1)
        .max(axis=1)
        .fillna(0)
    )
    print("✅ Criminal history features created")
else:
    print("⚠️  Criminal history features already exist — skipping")

# ---------------------------------------------------------------
# APPREHENSION METHOD FEATURES
# ---------------------------------------------------------------
if 'is_CAP_arrest' not in df.columns and 'Apprehension Method' in df.columns:
    df['is_CAP_arrest'] = df['Apprehension Method'].str.contains(
        'CAP', na=False).astype(int)
    df['is_287g_arrest'] = df['Apprehension Method'].str.contains(
        '287', na=False).astype(int)
    print(f"✅ CAP arrest rate: {df['is_CAP_arrest'].mean():.1%}")
    print(f"✅ 287(g) arrest rate: {df['is_287g_arrest'].mean():.1%}")
else:
    print("⚠️  Apprehension method features already exist or column missing — skipping")

# ---------------------------------------------------------------
# DEMOGRAPHIC FEATURES
# ---------------------------------------------------------------
if 'is_hispanic' not in df.columns and 'Ethnicity' in df.columns:
    df['is_hispanic'] = df['Ethnicity'].map(
        {'Hispanic Origin': 1, 'Not of Hispanic Origin': 0}
    )
    print(f"✅ Hispanic rate (among known): {df['is_hispanic'].mean():.1%}")
else:
    print("⚠️  is_hispanic already exists or Ethnicity missing — skipping")

if 'is_male' not in df.columns and 'Gender' in df.columns:
    df['is_male'] = df['Gender'].map(
        {'Male': 1, 'Female': 0, 'Unknown': np.nan}
    )
    print(f"✅ Male rate: {df['is_male'].mean():.1%}")
else:
    print("⚠️  is_male already exists or Gender missing — skipping")

if 'is_PWA' not in df.columns and 'Entry Status' in df.columns:
    df['is_PWA'] = df['Entry Status'].str.contains(
        'PWA', na=False).astype(int)
    print(f"✅ PWA rate: {df['is_PWA'].mean():.1%}")
else:
    print("⚠️  is_PWA already exists or Entry Status missing — skipping")

# ---------------------------------------------------------------
# ADMINISTRATION ENCODING
# ---------------------------------------------------------------
if 'administration_encoded' not in df.columns:
    admin_map = {'Obama': 0, 'Biden': 1, 'Trump': 2}
    df['administration_encoded'] = df['administration'].map(admin_map)
    print("✅ Administration encoded")
else:
    print("⚠️  administration_encoded already exists — skipping")

print(f"\nStep 8 complete. Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

⚠️  Date columns already processed or not present — skipping


/var/folders/q0/89dmr2hd7dj4vckq7lmx8ltw0000gn/T/ipykernel_29154/2722928329.py:26: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,21,22,23,29,30,31,32,33,34,35,36,37,38,39,40,42,45,46,47,48,49,50,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,69,70,71,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,90,91,92,93,94,96,97,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("/Users/sabrinkulatein/Downloads/ICE_Master_Clean.csv")


df rows: 2,299,497 | df_raw_aligned rows: 2,299,497
✅ Criminal history features created
✅ CAP arrest rate: 28.5%
✅ 287(g) arrest rate: 2.4%
✅ Hispanic rate (among known): 91.9%
✅ Male rate: 88.0%
✅ PWA rate: 50.7%
✅ Administration encoded

Step 8 complete. Dataset shape: (2299497, 38)
Columns: ['Apprehension Method', 'RCA_AOR', 'RCA_DCO', 'RISK_TO_PUBLIC_SAFETY', 'RISK_OF_FLIGHT', 'SPECIAL_VULNERABILITY', 'RCA_RECOMMENDATION', 'RCA_FINAL_DECISION', 'SPEC_VULN_VER', 'fiscal_year_code', 'fiscal_quarter_name', 'Detention Facility', 'Marital', 'Gender', 'Ethnicity', 'Birth Year', 'Entry Status', 'Case Status', 'Case Category', 'Case Threat Level', 'Charge', 'Birth Country', 'Citizenship Country', 'year', 'administration', 'age_at_arrest', 'target', 'days_to_book_in', 'apprehension_month', 'apprehension_year', 'any_criminality_recorded', 'max_criminality_severity', 'is_CAP_arrest', 'is_287g_arrest', 'is_hispanic', 'is_male', 'is_PWA', 'administration_encoded']


### Step 9 — Impute Missing Values
After feature engineering we drop the original raw columns that we replaced with cleaner engineered versions — Apprehension Method, Ethnicity, Gender, and Entry Status — since we no longer need them. We then fill in remaining missing values. For numeric columns we use the median rather than the mean because the median is not affected by extreme outliers. For categorical text columns we fill with the string 'Unknown', which honestly represents the fact that the information simply was not recorded rather than pretending it belongs to any real category. We leave the administration column untouched because we need it as a text label for splitting the data in Step 11.


In [ ]:
# Step 9 — Impute Missing Values

# Drop the original raw columns we replaced with engineered versions in Step 8
# We no longer need these since we extracted what we needed from them
cols_to_replace = [
    'Apprehension Method',  # replaced by is_CAP_arrest, is_287g_arrest
    'Ethnicity',            # replaced by is_hispanic
    'Gender',               # replaced by is_male
    'Entry Status',         # replaced by is_PWA
]
df = df.drop(columns=[c for c in cols_to_replace if c in df.columns])

# Separate numeric and categorical columns
# We exclude 'target' from imputation and 'administration' from encoding
# because we need both intact for later steps
num_cols = [c for c in df.select_dtypes(include='number').columns
            if c != 'target']
cat_cols = [c for c in df.select_dtypes(include='object').columns
            if c != 'administration']

print(f"Numeric columns to impute ({len(num_cols)}): {num_cols}")
print(f"Categorical columns to impute ({len(cat_cols)}): {cat_cols}")

# Fill numeric missing values with the column median
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical missing values with 'Unknown'
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Confirm no missing values remain
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if len(remaining_missing) == 0:
    print("\n✅ All missing values filled — no NaNs remaining")
else:
    print(f"\n⚠️  Missing values still present:")
    print(remaining_missing)

Numeric columns to impute (17): ['SPEC_VULN_VER', 'fiscal_year_code', 'Birth Year', 'Case Threat Level', 'year', 'age_at_arrest', 'days_to_book_in', 'apprehension_month', 'apprehension_year', 'any_criminality_recorded', 'max_criminality_severity', 'is_CAP_arrest', 'is_287g_arrest', 'is_hispanic', 'is_male', 'is_PWA', 'administration_encoded']
Categorical columns to impute (15): ['RCA_AOR', 'RCA_DCO', 'RISK_TO_PUBLIC_SAFETY', 'RISK_OF_FLIGHT', 'SPECIAL_VULNERABILITY', 'RCA_RECOMMENDATION', 'RCA_FINAL_DECISION', 'fiscal_quarter_name', 'Detention Facility', 'Marital', 'Case Status', 'Case Category', 'Charge', 'Birth Country', 'Citizenship Country']

✅ All missing values filled — no NaNs remaining


### Step 10 — Encode Categorical Variables
Machine learning models require all inputs to be numbers — they cannot work with text directly. For categorical columns with more than 20 unique values (high cardinality), like Detention Facility or Charge, we use frequency encoding, which replaces each category with how often it appears in the dataset as a proportion. This avoids creating hundreds of new columns. For columns with 20 or fewer unique values we use one-hot encoding, which creates a separate 0/1 column for each category. Note that Case Threat Level is already numeric and ordinal so we leave it as-is and it does not get encoded here.

In [ ]:
# Step 10 — Encode Categorical Variables

# Identify columns that still need encoding
# We skip 'administration' because we need it as a text label for Step 11
cat_cols_encode = [c for c in df.select_dtypes('object').columns
                   if c != 'administration']

print("Columns to encode:")
for c in cat_cols_encode:
    print(f"  {c}: {df[c].nunique()} unique values")

# --- High cardinality (>20 unique values): frequency encode ---
# Instead of creating hundreds of dummy columns, we replace each
# category with how often it appears in the dataset as a proportion.
# For example if 'Mexico' appears in 40% of rows, it becomes 0.40
high_card = [c for c in cat_cols_encode if df[c].nunique() > 20]
for col in high_card:
    freq = df[col].value_counts(normalize=True)
    df[col + '_freq'] = df[col].map(freq)
    df = df.drop(columns=[col])
    print(f"\n✅ Frequency encoded: {col} ({freq.shape[0]} unique values)")

# --- Low cardinality (<=20 unique values): one-hot encode ---
# Creates a separate 0/1 column for each category.
# drop_first=True drops one category per column to avoid
# perfect multicollinearity (the "dummy variable trap")
low_card = [c for c in df.select_dtypes('object').columns
            if c != 'administration']

print(f"\nOne-hot encoding: {low_card}")
df = pd.get_dummies(df, columns=low_card, drop_first=True)

print(f"\n✅ Encoding complete. Total columns: {df.shape[1]}")
print(f"\nAll columns after encoding:")
print(df.columns.tolist())

Columns to encode:
  RCA_AOR: 27 unique values
  RCA_DCO: 222 unique values
  RISK_TO_PUBLIC_SAFETY: 4 unique values
  RISK_OF_FLIGHT: 4 unique values
  SPECIAL_VULNERABILITY: 77 unique values
  RCA_RECOMMENDATION: 13 unique values
  RCA_FINAL_DECISION: 11 unique values
  fiscal_quarter_name: 5 unique values
  Detention Facility: 998 unique values
  Marital: 6 unique values
  Case Status: 13 unique values
  Case Category: 30 unique values
  Charge: 163 unique values
  Birth Country: 195 unique values
  Citizenship Country: 189 unique values

✅ Frequency encoded: RCA_AOR (27 unique values)

✅ Frequency encoded: RCA_DCO (222 unique values)

✅ Frequency encoded: SPECIAL_VULNERABILITY (77 unique values)

✅ Frequency encoded: Detention Facility (998 unique values)

✅ Frequency encoded: Case Category (30 unique values)

✅ Frequency encoded: Charge (163 unique values)

✅ Frequency encoded: Birth Country (195 unique values)

✅ Frequency encoded: Citizenship Country (189 unique values)

One-hot

### Step 11 — Create Modeling Splits
This is the final setup step before modeling. We create three separate datasets: one global dataset using all records across all three administrations, one for Obama-era records only, and one for Trump-era records only. Biden records stay in the global dataset to improve overall model training volume, but we do not create a separate Biden split because our paper's formal hypotheses are specifically about Obama versus Trump. Comparing feature importances between the Obama and Trump models is how we directly test Hypothesis 3 — that policy shifts meaningfully reshape how deportation decisions are structured.

In [ ]:
# Step 11 — Create Modeling Splits

# --- Global dataset: all administrations ---
X_global = df.drop(columns=['administration', 'target'])
y_global  = df['target'].astype(int)

# --- Obama-only subset ---
obama_mask = df['administration'] == 'Obama'
X_obama = df[obama_mask].drop(columns=['administration', 'target'])
y_obama  = df[obama_mask]['target'].astype(int)

# --- Trump-only subset ---
trump_mask = df['administration'] == 'Trump'
X_trump = df[trump_mask].drop(columns=['administration', 'target'])
y_trump  = df[trump_mask]['target'].astype(int)

In [ ]:
# --- Final summary ---
print("=" * 50)
print("PIPELINE COMPLETE — READY FOR MODELING")
print("=" * 50)
print(f"Global:  {X_global.shape[0]:,} rows, {X_global.shape[1]} features")
print(f"Obama:   {X_obama.shape[0]:,} rows,  {X_obama.shape[1]} features")
print(f"Trump:   {X_trump.shape[0]:,} rows,  {X_trump.shape[1]} features")
print(f"\nTarget balance (global):")
print(y_global.value_counts(normalize=True).round(3))
print(f"\nObama target balance:")
print(y_obama.value_counts(normalize=True).round(3))
print(f"\nTrump target balance:")
print(y_trump.value_counts(normalize=True).round(3))
print(f"\nAll features going into the model:")
print(X_global.columns.tolist())

PIPELINE COMPLETE — READY FOR MODELING
Global:  2,299,497 rows, 74 features
Obama:   933,387 rows,  74 features
Trump:   1,016,206 rows,  74 features

Target balance (global):
target
1    0.731
0    0.269
Name: proportion, dtype: float64

Obama target balance:
target
1    0.805
0    0.195
Name: proportion, dtype: float64

Trump target balance:
target
1    0.751
0    0.249
Name: proportion, dtype: float64

All features going into the model:
['SPEC_VULN_VER', 'fiscal_year_code', 'Birth Year', 'Case Threat Level', 'year', 'age_at_arrest', 'days_to_book_in', 'apprehension_month', 'apprehension_year', 'any_criminality_recorded', 'max_criminality_severity', 'is_CAP_arrest', 'is_287g_arrest', 'is_hispanic', 'is_male', 'is_PWA', 'administration_encoded', 'RCA_AOR_freq', 'RCA_DCO_freq', 'SPECIAL_VULNERABILITY_freq', 'Detention Facility_freq', 'Case Category_freq', 'Charge_freq', 'Birth Country_freq', 'Citizenship Country_freq', 'RISK_TO_PUBLIC_SAFETY_Low', 'RISK_TO_PUBLIC_SAFETY_Medium', 'RISK_